# Can you build the growth team's Monday table alone, end to end?

**Week 2, Thursday. The escalated case, unguided, 50 minutes.** The brief is
`exercises/unguided/C2_W02_D04_escalated_case_STUDENT.md`, and this notebook is where its five
parts run. Each step carries one or two lettered choices written as comments above a
`__TODOn__` placeholder: replace the placeholder with the code of the option you choose and run
the step's check. Run as shipped, the notebook stops at `__TODO1__` with a `NameError`, which is
how it is meant to start.

> **The client asks.** "One table, one row per customer, refreshed every Monday: how recently,
> how often, how much, the segment, whether the monsoon sale reached them, and the two flags we
> act on. And one view of it by month that we can put on a slide."
>
> The growth team, Kalpa Retail

**What the morning established, in one line each.** The customer list is the table's spine,
and a customer with no orders gets a frequency and spend of 0. A merge is a join, and
`validate="one_to_one"` stops it if a customer repeats. A feed that sends a customer twice is
handled by a rule, one row per customer, the first date the sale reached them. `pivot_table`
needs its `aggfunc` said out loud. Recency is counted to the data's own last date. The table
ships only when every guard passes.

**The two flags.** *Lapsed:* no order in the 60 days to the table's as-of date; a customer who
never ordered is not lapsed, since there is nothing to win back. *Falling:* Wednesday's rule,
spend lower in August than in July and lower again in September than in August, each reading a
real calendar month after the one before, so a month with no order breaks the run.

> **Kavya's review.** "It ships when the checks pass, not when it runs. Post your letters and
> the four numbers the last cell prints."

**Setup.** The warehouse read, the campaign platform's feed, and the data's last date under a name the steps use.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit
import pandas as pd

ENG = kit.engine()                  # the Kalpa warehouse in Postgres, read only
orders = pd.read_sql(
    "SELECT order_id, customer_id, order_date, quarter, channel, amount, status FROM orders",
    ENG, parse_dates=["order_date"])
customers = pd.read_sql("SELECT customer_id, segment, city FROM customers", ENG)
print(f"{len(orders):,} orders and {len(customers):,} customers read from the warehouse")

exposure = pd.read_csv(kit.data_dir() / "C2_W02_D04_exposure_STUDENT.csv", parse_dates=["exposed_date"])
print("the campaign platform's exposure feed, columns:", list(exposure.columns))

In [ ]:
kit.side_by_side(
    kit.ladder(["the spine and three numbers", "the exposure, validated", "the two flags",
                "one reshaped view", "one run, with guards"], show=False),
    kit.vflow(["the warehouse and the feed", "one table, 340 rows", "a slide, and a CSV for Friday"],
              title="Five steps, each with its checks", show=False),
)

## Step 1. Does the table hold every customer, with the three numbers each?

Where it is used at work: every table a growth team acts on starts from the list of the people
it acts on, never from their transactions.

In [ ]:
rfm = (orders.groupby("customer_id")
             .agg(last_order=("order_date", "max"), frequency=("order_id", "count"),
                  spend=("amount", "sum"))
             .reset_index())

# TODO 1. Which frame is the spine of a table that must hold every customer?
#   a) orders
#   b) rfm
#   c) customers
#   d) exposure.drop_duplicates("customer_id")
table = __TODO1__.merge(rfm, on="customer_id", how="left", validate="one_to_one")

# TODO 2. What goes in the frequency of a customer who never ordered?
#   a) table["frequency"]
#   b) table["frequency"].fillna(0).astype("int64")
#   c) table["frequency"].fillna(table["frequency"].median())
#   d) table["frequency"].dropna()
table = table.assign(frequency=__TODO2__, spend=table["spend"].fillna(0))
print(f"{len(table)} rows")

In [ ]:
in_list = kit.sql("SELECT count(*) AS n FROM customers")[0]["n"]
kit.check("one row for every customer on the list", len(table) == in_list, f"{len(table)} rows")
kit.check("spend adds back to the warehouse", table["spend"].sum() == orders["amount"].sum())
kit.check("frequency is a whole number with no gaps", table["frequency"].dtype == "int64")

## Step 2. Which customers did the sale reach, one row each?

Where it is used at work: any feed from another team's system is checked against the promise
it makes before it is joined to a table other people act on.

In [ ]:
# TODO 3. Which argument keeps one row per customer, their first exposure?
#   a) keep="last"
#   b) keep=False
#   c) keep="first"
#   d) ignore_index=True
first_touch = (exposure.sort_values("exposed_date")
                       .drop_duplicates("customer_id", __TODO3__)[["customer_id", "exposed_date"]])

# TODO 4. Which promise guards a table that must stay one row per customer?
#   a) "one_to_many"
#   b) "many_to_many"
#   c) None
#   d) "one_to_one"
table = table.merge(first_touch, on="customer_id", how="left", validate=__TODO4__)
table = table.assign(reached=table["exposed_date"].notna())
print(f"{len(table)} rows after the merge")

In [ ]:
kit.check("still one row per customer after the merge", len(table) == in_list and table["customer_id"].is_unique)
kit.check("spend did not move in the merge", table["spend"].sum() == orders["amount"].sum())
kit.check("every customer the feed names is marked reached, once",
          int(table["reached"].sum()) == exposure["customer_id"].nunique())
reach = table.groupby("segment")["reached"].sum().astype(int)
kit.columns(reach.index.tolist(), [("reached by the sale", reach.tolist())],
            title="Reached customers per segment, one row each")

## Step 3. Who has gone quiet, and whose monthly spend is falling?

Where it is used at work: a retention team's weekly list is two flags, one for customers who
stopped and one for customers who are slowing, and each has to mean the same thing every week.

In [ ]:
# TODO 5. Recency is counted to which date?
#   a) pd.Timestamp.today()
#   b) orders["order_date"].max()
#   c) orders["order_date"].min()
#   d) pd.Timestamp.today().normalize()
AS_OF = __TODO5__
table = table.assign(as_of=AS_OF, recency_days=(AS_OF - table["last_order"]).dt.days)

# TODO 6. Which condition flags a lapsed customer?
#   a) table["recency_days"] > 60
#   b) table["frequency"] == 0
#   c) table["recency_days"].isna()
#   d) (pd.Timestamp.today() - table["last_order"]).dt.days > 60
table = table.assign(lapsed=__TODO6__)

monthly = (orders.assign(month_num=orders["order_date"].dt.year * 12 + orders["order_date"].dt.month)
                 .groupby(["customer_id", "month_num"], as_index=False)["amount"].sum()
                 .rename(columns={"amount": "spend"})
                 .sort_values(["customer_id", "month_num"]))
g = monthly.groupby("customer_id")
# TODO 7. The spend of the same customer's previous reading is:
#   a) monthly["spend"].shift(1)
#   b) monthly.groupby("customer_id")["spend"].shift(1)
#   c) monthly.groupby("month_num")["spend"].shift(1)
#   d) monthly.groupby(["customer_id", "month_num"])["spend"].shift(1)
monthly = monthly.assign(spend_1=__TODO7__, spend_2=g["spend"].shift(2),
                         month_num_2=g["month_num"].shift(2))
SEPTEMBER = 2026 * 12 + 9

# TODO 8. Which test keeps a fall only when the two readings before September are July and August?
#   a) monthly["month_num"] - monthly["month_num_2"] == 2
#   b) monthly["month_num_2"].notna()
#   c) monthly["month_num"] - monthly["month_num_2"] >= 2
#   d) monthly["month_num"] - monthly["month_num_2"] <= 3
real_months = __TODO8__
falling = monthly[(monthly["month_num"] == SEPTEMBER) & real_months
                  & (monthly["spend"] < monthly["spend_1"]) & (monthly["spend_1"] < monthly["spend_2"])]
table = table.assign(falling=table["customer_id"].isin(falling["customer_id"]))
print("both flags are set")

In [ ]:
kit.check("the smallest recency is 0 days", table["recency_days"].min() == 0)
by_sql = int(kit.sql("""WITH last  AS (SELECT customer_id, max(order_date) AS last_order FROM orders GROUP BY customer_id),
     as_of AS (SELECT max(order_date) AS d FROM orders)
SELECT count(*) AS win_back
FROM   last, as_of
WHERE  as_of.d - last.last_order > 60;""")[0]["win_back"])
kit.check("the win-back list matches the warehouse's own count", int(table["lapsed"].sum()) == by_sql)
wednesday = {r["customer_id"] for r in kit.sql("""WITH monthly AS (
         SELECT customer_id, date_trunc('month', order_date)::date AS month, sum(amount) AS spend
         FROM   orders
         GROUP  BY customer_id, date_trunc('month', order_date)),
     lagged AS (
         SELECT customer_id, month, spend,
                lag(spend, 1) OVER w AS spend_before, lag(spend, 2) OVER w AS spend_two_before,
                lag(month, 1) OVER w AS month_before, lag(month, 2) OVER w AS month_two_before
         FROM   monthly
         WINDOW w AS (PARTITION BY customer_id ORDER BY month))
SELECT customer_id
FROM   lagged
WHERE  month = DATE '2026-09-01'
  AND  month_before = DATE '2026-08-01' AND month_two_before = DATE '2026-07-01'
  AND  spend < spend_before AND spend_before < spend_two_before;""")}
kit.check("pandas flags exactly the customers Wednesday's calendar-checked LAG flags",
          set(table.loc[table["falling"], "customer_id"]) == wednesday)
kit.check("both flags are booleans", table["lapsed"].dtype == bool and table["falling"].dtype == bool)
lap = table.groupby("segment")["lapsed"].sum().astype(int)
kit.columns(lap.index.tolist(), [("lapsed, as of the data's last date", lap.tolist())],
            title="The win-back list per segment")

## Step 4. How does spend move month by month in each segment?

Where it is used at work: the slide a growth review opens on is a trend by segment, and the
trend is only as right as the aggregation behind each point.

In [ ]:
seg_orders = (orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
                    .assign(month=orders["order_date"].dt.to_period("M").astype(str)))
# TODO 9. Which call puts months on the rows, one column per segment, and adds the orders in each cell?
#   a) seg_orders.pivot_table(index="segment", columns="month", values="amount", aggfunc="sum")
#   b) seg_orders.pivot_table(index="month", columns="segment", values="amount", aggfunc="sum")
#   c) seg_orders.pivot_table(index="month", columns="segment", values="amount")
#   d) seg_orders.pivot_table(index="month", columns="segment", values="amount", aggfunc="count")
view = __TODO9__
print("view shape:", view.shape)

In [ ]:
kit.check("the view adds back to the warehouse", view.values.sum() == orders["amount"].sum())
kit.check("one row per month, April to September", list(view.index) == sorted(seg_orders["month"].unique()))
kit.line(list(view.index), [(s, view[s].tolist(), "lit" if s == "Retail-Plus" else "")
                            for s in ["Retail-Core", "Retail-Plus", "Student"]], fmt=kit.rupees,
         title="The slide: consumer spend by month and segment")

## Step 5. Will the table rebuild itself next Monday and stop if something breaks?

Where it is used at work: a scheduled job that feeds a campaign is trusted only when it
refuses to write a table that fails its checks.

In [ ]:
def guard(t):
    # TODO 10. Which guard stops a table that has stopped being one row per customer?
    #   a) t["customer_id"].is_unique
    #   b) len(t) > 0
    #   c) t["spend"].sum() > 0
    #   d) t["customer_id"].notna().all()
    assert __TODO10__, "the customer table is no longer one row per customer"
    assert len(t) == in_list, "the table no longer holds every customer on the list"
    assert t["spend"].sum() == orders["amount"].sum(), "spend no longer adds to the warehouse"
    # TODO 11. Which guard catches recency counted to the wrong date?
    #   a) t["recency_days"].min() == 0
    #   b) t["recency_days"].max() <= 180
    #   c) t["recency_days"].notna().all()
    #   d) t["as_of"].nunique() == 1
    assert __TODO11__, "recency was not counted to the data's last date"
    return t

run_1 = guard(table.copy())
run_2 = guard(table.copy())
print(len(run_1), "rows passed every guard")

In [ ]:
kit.check("two guarded runs give the same table", run_1.equals(run_2))
broken = pd.concat([table, table.iloc[[150]]])
with kit.expect_error() as stopped:
    guard(broken)
kit.check("a repeated customer stops the run", stopped.name == "AssertionError")
out = pathlib.Path("output"); out.mkdir(exist_ok=True)
run_1.to_csv(out / "C2_W02_D04_customer_table_STUDENT.csv", index=False)
kit.stats([(len(run_1), "customers", "one row each"), (kit.rupees(run_1["spend"].sum()), "spend", "adds to the warehouse"),
           (int(run_1["reached"].sum()), "reached by the sale", "first exposure per customer"),
           (int(run_1["lapsed"].sum()), "on the win-back list", "as of the data's last date")])
kit.flow(["warehouse + feed", "the table, guarded", "output/ CSV\nfor Friday"], lit=1,
         title="The Monday table, one guarded run")
kit.check_summary()

**Post:** your eleven letters in order, and the four numbers above. The CSV in `output/` is
what you bring to Friday.